In [1]:
# Install dependencies

%pip install anthropic python-dotenv


[notice] A new release of pip is available: 26.1.2 -> 26.2.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [2]:
"""
CONTROLLING MODEL OUTPUT
-> It allows for more dynamic and interactive applications.

TWO TECHNIQUES:
-> prefilled assistant messages: Guide the model's behavior by providing example assistant messages in the conversation history.
-> stop sequences: Define specific sequences of text that, when generated by the model, will signal it to stop generating further content.
"""

# Load environment variables
from dotenv import load_dotenv

load_dotenv()

True

In [3]:
# Create an API client
from anthropic import Anthropic

client = Anthropic()

model = "claude-sonnet-4-5"

In [13]:
# Helpers
def add_message(messages, content, role):
    message = {"role": role, "content": content}
    messages.append(message)

"""
Stop Sequences
-> Force Claude to stop generating further content when it produces specific sequences of text.
-> Useful for controlling the length and format of the output.
"""

def chat(messages, system=None, temperature=1.0, stop_sequences=[]):
    params = {
        "model": model,
        "max_tokens": 1000,
        "messages": messages,
        "temperature": temperature,
        "stop_sequences": stop_sequences # } </output>
    }

    if system:
        params["system"] = system

    message = client.messages.create(**params)

    return message

In [5]:
"""
Message Prefilling
-> Provide the start of an assitant message as your last message
-> Claude will continue the response from there
-> This will greatly steer Claude's response
"""
messages = []

add_message(messages, 
            "Is tea or coffee better at breakfast?", 
            role="user")

# Prefill assistant message
add_message(messages, 
            # "Coffee is better because", 
            # "Tea is better because", 
            # "Both tea and coffee have their own unique benefits for breakfast. ",
            'Neigther tea nor coffee is better because',
            role="assistant")

answer = chat(messages)

answer

', it\'s all about personal preference! Here are some factors to consider:\n\n**Coffee might be better if you:**\n- Need a stronger caffeine boost (coffee has about 2-3x more caffeine)\n- Enjoy bolder, richer flavors\n- Want something that pairs well with sweet breakfast foods\n\n**Tea might be better if you:**\n- Prefer a gentler caffeine lift\n- Want more variety in flavors (black, green, herbal, etc.)\n- Are sensitive to caffeine or acid\n- Enjoy lighter, more subtle tastes\n\n**Health-wise**, both have benefits - antioxidants, potential metabolism support, and mental alertness. The "best" choice is really the one you enjoy and that makes you feel good.\n\nWhat matters most is staying hydrated and starting your day with something you actually like!'

In [14]:
"""
Stop Sequences
"""
messages = []

add_message(messages, 
            "Count from 1 to 10", 
            role="user")

answer = chat(messages , stop_sequences=[", 5"])

answer




Message(id='msg_011CeCE6e9kS3Pg4wPD9i19W', container=None, content=[TextBlock(citations=None, text='1, 2, 3, 4', type='text')], model='claude-sonnet-4-5-20250929', role='assistant', stop_details=None, stop_reason='stop_sequence', stop_sequence=', 5', type='message', usage=Usage(cache_creation=CacheCreation(ephemeral_1h_input_tokens=0, ephemeral_5m_input_tokens=0), cache_creation_input_tokens=0, cache_read_input_tokens=0, inference_geo='not_available', input_tokens=15, output_tokens=14, output_tokens_details=None, server_tool_use=None, service_tier='standard'))

In [7]:
"""
Structured data

-> Ask Clude to output data in a structured format like JSON, CSV, bullet points, etc.
"""

messages = []

add_message(messages, 
            "List 5 popular programming languages and their main use cases in JSON format.", 
            role="user")

# prefill assistant message
add_message(messages, "```json", role="assistant"),

# Stop sequence to end JSON block
answer = chat(messages, stop_sequences=["```"])

answer

'\n[\n  {\n    "language": "Python",\n    "main_use_cases": [\n      "Data science and machine learning",\n      "Web development (Django, Flask)",\n      "Automation and scripting",\n      "Scientific computing"\n    ]\n  },\n  {\n    "language": "JavaScript",\n    "main_use_cases": [\n      "Front-end web development",\n      "Back-end development (Node.js)",\n      "Mobile app development (React Native)",\n      "Interactive web applications"\n    ]\n  },\n  {\n    "language": "Java",\n    "main_use_cases": [\n      "Enterprise applications",\n      "Android mobile development",\n      "Large-scale systems",\n      "Web applications (Spring framework)"\n    ]\n  },\n  {\n    "language": "C++",\n    "main_use_cases": [\n      "Game development",\n      "System/operating system programming",\n      "High-performance applications",\n      "Embedded systems"\n    ]\n  },\n  {\n    "language": "SQL",\n    "main_use_cases": [\n      "Database management",\n      "Data querying and manipul

In [8]:
import json

data = json.loads(answer.strip())
data

[{'language': 'Python',
  'main_use_cases': ['Data science and machine learning',
   'Web development (Django, Flask)',
   'Automation and scripting',
   'Scientific computing']},
 {'language': 'JavaScript',
  'main_use_cases': ['Front-end web development',
   'Back-end development (Node.js)',
   'Mobile app development (React Native)',
   'Interactive web applications']},
 {'language': 'Java',
  'main_use_cases': ['Enterprise applications',
   'Android mobile development',
   'Large-scale systems',
   'Web applications (Spring framework)']},
 {'language': 'C++',
  'main_use_cases': ['Game development',
   'System/operating system programming',
   'High-performance applications',
   'Embedded systems']},
 {'language': 'SQL',
  'main_use_cases': ['Database management',
   'Data querying and manipulation',
   'Data analysis',
   'Backend data operations']}]

# Structured Data Exercise

- Use message prefilling and stop sequence *only* to get three different commands in a single response
- There shouldn't be any comments or explanation
- Hint: message prefelling isn't limited to just chracters like ```

In [9]:
messages = []

prompt = """
Generate three different sample AWS CLI commands. Each should be very short.
"""

add_message(messages, prompt, role="user")

text = chat(messages)
text.strip()

'Here are three short AWS CLI commands:\n\n```bash\naws s3 ls\n\naws ec2 describe-instances\n\naws iam list-users\n```'

In [10]:
from IPython.display import Markdown

Markdown(text)

Here are three short AWS CLI commands:

```bash
aws s3 ls

aws ec2 describe-instances

aws iam list-users
```

In [11]:
# SOLUTION
messages = []

prompt = """
Generate three different sample AWS CLI commands. Each should be very short.
"""
add_message(messages, prompt, role="user")

# prefill assistant message
add_message(messages, "Here are three short AWS CLI commands:```bash", role="assistant"),

# Stop sequence to end JSON block
text = chat(messages, stop_sequences=["```"])

text.strip()



'aws s3 ls\n\naws ec2 describe-instances\n\naws iam list-users'

# Prefilling + Stop Sequences Across Multiple Formats (Single Interaction)

The single-request attempts above ask Claude for "bash, xml and json" all at once with one generic prefill (` ```code `). Since the prefill doesn't commit to a specific format, Claude doesn't reliably fence each block, and one `stop_sequences=["```"]` only catches the *first* fence — so the response gets cut off after the intro line.

The fix, still in **one** `chat()` call:

- **Prefill** with ` ```bash ` so Claude commits to starting the very first block in the right format, and is explicitly told the exact order (bash, then xml, then json).
- Instead of stopping at every ` ``` ` (which would cut off after the first block), the prompt asks Claude to print a unique marker (`<<END>>`) once all three blocks are done, and that marker is the **stop sequence** — so generation runs through all three blocks and stops right after.
- Since the prefill text itself isn't echoed back in the response, it's re-attached before parsing out the three fenced blocks with a regex.

In [ ]:
import re

messages = []
prefill = "```bash"

prompt = """
Generate one short sample output for storing a user configuration
(username, theme, notifications_enabled) in each of these formats, in this
exact order: bash, xml, json.

Wrap each snippet in its own fenced code block (```bash ... ```, then
```xml ... ```, then ```json ... ```). No explanations or extra text
between blocks. After the json code block, output the exact marker <<END>>.
"""
add_message(messages, prompt, role="user")

# Prefill commits Claude to starting directly with the bash block
add_message(messages, prefill, role="assistant")

# Stop sequence ends generation right after the final block, not after each fence
text = chat(messages, stop_sequences=["<<END>>"])

# The prefill isn't echoed back in the response, so re-attach it before parsing
full_response = prefill + text

samples = dict(re.findall(r"```(\w+)\n(.*?)```", full_response, re.DOTALL))

for fmt, snippet in samples.items():
    print(f"--- {fmt.upper()} ---")
    print(snippet.strip())
    print()

--- BASH ---
USERNAME="john_doe"
THEME="dark"
NOTIFICATIONS_ENABLED=true

--- XML ---
<?xml version="1.0" encoding="UTF-8"?>
<user_config>
    <username>john_doe</username>
    <theme>dark</theme>
    <notifications_enabled>true</notifications_enabled>
</user_config>

--- JSON ---
{
    "username": "john_doe",
    "theme": "dark",
    "notifications_enabled": true
}



Agentic AI
1000 todos -> llm  [mcp, got to a tool 1000 todos ]

1000 todos -> tool 